# 01 — AzSL data exploration

Inspect the AzSL-Text descriptions, the seen/unseen split, and the per-gloss video counts.

**Inputs**
- `data/azsld/descriptions.json` — `{gloss_id: 'Azerbaijani description'}`
- `data/azsld/splits/seen_glosses.txt` / `unseen_glosses.txt`
- `data/azsld/videos/{gloss_id}/*.mp4` (optional, used for counts only)

If the real dataset is not present, this notebook falls back to a minimal synthetic example.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    pass
elif (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data.preprocessing import DescriptionLoader
from src.data.dataset import discover_videos
from src.data.splits import load_split
ROOT

In [ ]:
DESC_PATH = ROOT / 'data/azsld/descriptions.json'
SPLITS_DIR = ROOT / 'data/azsld/splits'
VIDEO_DIR  = ROOT / 'data/azsld/videos'

if DESC_PATH.exists():
    desc_loader = DescriptionLoader(DESC_PATH)
    descriptions = dict(desc_loader.items())
else:
    print('No real AzSL-Text found — using a synthetic 5-gloss example.')
    descriptions = {
        'ata':   'Baş barmaq alına toxunur, digər barmaqlar açıq və yuxarı yönəlmişdir.',
        'ana':   'Baş barmaq çənəyə toxunur.',
        'su':    'Üç barmaq dodağa yaxınlaşır və açılır.',
        'ev':    'İki əl ev şəklində birləşir.',
        'kitab': 'İki əl açıq kitab kimi açılır.',
    }
len(descriptions), list(descriptions.items())[:3]

## Description length distribution

Paper Sec. III-A: a structured definition that names handshape, palm orientation,
movement, location, and non-manual markers. Very short descriptions tend to be
underspecified — the qualitative analysis in Sec. V-G traces failures to overly
generic descriptions.

In [ ]:
lengths = np.array([len(d.split()) for d in descriptions.values()])
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=min(30, len(lengths)), color='#1f4e79', alpha=0.85, edgecolor='white')
ax.axvline(lengths.mean(), color='#c0504d', ls='--', lw=2,
           label=f'mean = {lengths.mean():.1f} words')
ax.set_xlabel('Words per description')
ax.set_ylabel('Count')
ax.set_title('Description length (words)')
ax.legend()
plt.tight_layout(); plt.show()

## Seen / unseen split

In [ ]:
seen_path = SPLITS_DIR / 'seen_glosses.txt'
unseen_path = SPLITS_DIR / 'unseen_glosses.txt'
if seen_path.exists() and unseen_path.exists():
    seen = load_split(seen_path)
    unseen = load_split(unseen_path)
    print(f'Seen:   {len(seen):>4} glosses (e.g. {seen[:5]})')
    print(f'Unseen: {len(unseen):>4} glosses (e.g. {unseen[:5]})')
    print(f'Overlap: {len(set(seen) & set(unseen))} (must be 0)')
else:
    from src.data.splits import build_random_split
    seen, unseen = build_random_split(list(descriptions.keys()), num_unseen=1, seed=42)
    print('(Synthetic split.) Seen:', seen, 'Unseen:', unseen)

## Per-gloss video counts (if videos available)

In [ ]:
if VIDEO_DIR.exists():
    pairs = discover_videos(VIDEO_DIR)
    counts = {}
    for gloss, _ in pairs:
        counts[gloss] = counts.get(gloss, 0) + 1
    print(f'{len(pairs)} videos across {len(counts)} glosses '
          f'(min={min(counts.values())}, max={max(counts.values())}, '
          f'mean={np.mean(list(counts.values())):.1f})')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(list(counts.values()), bins=30, color='#4bacc6', alpha=0.85,
            edgecolor='white')
    ax.set_xlabel('Videos per gloss')
    ax.set_ylabel('Number of glosses')
    ax.set_title('Per-gloss video count distribution')
    plt.tight_layout(); plt.show()
else:
    print(f'No videos at {VIDEO_DIR} — skipping per-gloss count plot.')

## Sample-text inspection — prompt ensemble preview

In [ ]:
from src.data.preprocessing import PromptBuilder
pb = PromptBuilder()
first_gloss = next(iter(descriptions))
for i, prompt in enumerate(pb(descriptions[first_gloss]), 1):
    print(f'[{i}] {prompt}')